# Extended Python Data Cleaning Cookbook: Continuous Variable Analysis
This notebook expands upon the recipe **"Generating summary statistics for continuous variables"** from the *Python Data Cleaning Cookbook* (Chapter 3).

### Key Enhancements & Additions:
1. **Fallback Data Synthesizer**: Self-contained execution without external dependency breaks.
2. **Skewness, Kurtosis & Formal Normality Testing**: Quantifies non-normality using D'Agostino-Pearson tests.
3. **Automated IQR Outlier Identification**: Fences and isolates high-leverage extreme values.
4. **Data Transformation Pipeline**: Normalizes heavy tails using log transformations (`np.log1p`).
5. **Multi-panel Visual Suite**: Seaborn KDEs, Boxplots, and Q-Q plots.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette("muted")
%matplotlib inline

## Step 1: Data Ingestion & Structural Inspection

In [ ]:
def load_or_generate_data(filepath="data/covidtotals.csv"):
    if os.path.exists(filepath):
        df = pd.read_csv(filepath, parse_dates=['lastdate'])
        df.set_index("iso_code", inplace=True)
        return df
    else:
        print(f"File '{filepath}' not found. Generating synthetic dataset...")
        np.random.seed(42)
        n_samples = 210
        iso_codes = [f"ISO_{i:03d}" for i in range(1, n_samples + 1)]
        
        total_cases = np.random.lognormal(mean=8.5, sigma=2.2, size=n_samples).astype(int)
        total_deaths = (total_cases * np.random.beta(a=0.5, b=10, size=n_samples)).astype(int)
        population = np.random.lognormal(mean=15.5, sigma=1.8, size=n_samples)
        
        df = pd.DataFrame({
            'lastdate': pd.to_datetime('2020-06-01'),
            'location': [f"Country_{i}" for i in range(1, n_samples + 1)],
            'total_cases': total_cases,
            'total_deaths': total_deaths,
            'total_cases_pm': (total_cases / population) * 1e6,
            'total_deaths_pm': (total_deaths / population) * 1e6,
            'population': population,
            'pop_density': np.random.lognormal(mean=4, sigma=1.5, size=n_samples),
            'median_age': np.random.normal(loc=30, scale=8, size=n_samples).clip(15, 50),
            'gdp_per_capita': np.random.lognormal(mean=9.2, sigma=1.1, size=n_samples),
            'hosp_beds': np.random.gamma(shape=2, scale=1.5, size=n_samples)
        }, index=iso_codes)
        df.index.name = "iso_code"
        return df

covidtotals = load_or_generate_data()
print("Shape:", covidtotals.shape)
covidtotals.dtypes

## Step 2: Summary Statistics & Decile Breakdown

In [ ]:
numeric_cols = ['total_cases', 'total_deaths', 'total_cases_pm', 'total_deaths_pm', 'median_age', 'gdp_per_capita', 'hosp_beds']
display(covidtotals[numeric_cols].describe().T)

covidtotals[numeric_cols].quantile(np.arange(0.0, 1.1, 0.1)).T

## Enhancement 1: Statistical Skewness & Normality Diagnostics

In [ ]:
diag_results = []
for col in numeric_cols:
    s = covidtotals[col].dropna()
    skew_val = s.skew()
    kurt_val = s.kurtosis()
    stat, p_val = stats.normaltest(s)
    diag_results.append({
        'Variable': col,
        'Skewness': round(skew_val, 3),
        'Kurtosis': round(kurt_val, 3),
        'Normality Test Stat': round(stat, 3),
        'p-value': f"{p_val:.4e}",
        'Is Gaussian (p>0.05)': p_val > 0.05
    })

pd.DataFrame(diag_results)

## Enhancement 2: Automated Outlier Identification (IQR Fence)

In [ ]:
def detect_outliers_iqr(df, col):
    q25, q75 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q75 - q25
    lower, upper = q25 - 1.5 * iqr, q75 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"Column '{col}' - Found {len(outliers)} outliers outside range [{lower:.2f}, {upper:.2f}]")
    return outliers

outliers_cases = detect_outliers_iqr(covidtotals, 'total_cases')
outliers_cases[['location', 'total_cases', 'total_deaths', 'population']].head()

## Enhancement 3: Multi-panel Diagnostic Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Continuous Variable Diagnostic Suite: total_cases", fontsize=16, fontweight='bold')

# 1. Baseline Histogram
axes[0, 0].hist(covidtotals['total_cases'] / 1000, bins=15, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title("Baseline Histogram (x1000)")
axes[0, 0].set_xlabel("Cases (Thousands)")
axes[0, 0].set_ylabel("Count")

# 2. Log-scale Histogram + KDE Density
sns.histplot(covidtotals['total_cases'], kde=True, log_scale=True, ax=axes[0, 1], color='teal')
axes[0, 1].set_title("Log-scale Histogram with Kernel Density Estimate (KDE)")

# 3. Boxplot
sns.boxplot(x=covidtotals['total_cases'], ax=axes[1, 0], color='salmon')
axes[1, 0].set_title("Boxplot (Outlier Visualizer)")

# 4. Q-Q Plot
stats.probplot(np.log1p(covidtotals['total_cases'].dropna()), dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("Q-Q Plot (Log Transformed vs Gaussian Normal)")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()